In [6]:
import torch

checkpoint_path = 'data/checkpoint/chkpnt6000.pth'

# explicitly weights_only=False
model_params, first_iter = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

print(f"[INFO] Loaded checkpoint at iteration: {first_iter}")

print("\n[INFO] structure of model_params :")
for i, item in enumerate(model_params):
    if hasattr(item, 'shape'):
        print(f"  [{i}] Type: {type(item).__name__}, Shape: {item.shape}")
    elif isinstance(item, dict):
        print(f"  [{i}] Type: dict, Keys: {list(item.keys())}")
    elif isinstance(item, int) or isinstance(item, float):
        print(f"  [{i}] Type: {type(item).__name__}, Value: {item}")
    else:
        print(f"  [{i}] Type: {type(item).__name__}, Content: {item}")

is_4d = len(model_params) > 12
print("\n[INFO] Inferred Gaussian dimension:", "4D" if is_4d else "3D")


[INFO] Loaded checkpoint at iteration: 6000

[INFO] structure of model_params :
  [0] Type: int, Value: 3
  [1] Type: Parameter, Shape: torch.Size([705952, 3])
  [2] Type: Parameter, Shape: torch.Size([705952, 1, 3])
  [3] Type: Parameter, Shape: torch.Size([705952, 15, 3])
  [4] Type: Parameter, Shape: torch.Size([705952, 3])
  [5] Type: Parameter, Shape: torch.Size([705952, 4])
  [6] Type: Parameter, Shape: torch.Size([705952, 1])
  [7] Type: Tensor, Shape: torch.Size([705952])
  [8] Type: Tensor, Shape: torch.Size([705952, 1])
  [9] Type: Tensor, Shape: torch.Size([705952, 1])
  [10] Type: Tensor, Shape: torch.Size([705952, 1])
  [11] Type: dict, Keys: ['state', 'param_groups']
  [12] Type: float64, Shape: ()
  [13] Type: Parameter, Shape: torch.Size([705952, 1])
  [14] Type: Parameter, Shape: torch.Size([705952, 1])
  [15] Type: Parameter, Shape: torch.Size([705952, 4])
  [16] Type: bool, Value: True
  [17] Type: NoneType, Content: None
  [18] Type: int, Value: 2
  [19] Type: Param

In [21]:
import torch, numpy as np, pathlib, sys, warnings, pickle

ckpt = pathlib.Path("data/checkpoint/chkpnt6000.pth")
xyz_out  = ckpt.with_suffix(".xyz.txt")
stat_out = ckpt.with_suffix(".static_xyz.txt")

try:
    state = torch.load(ckpt, map_location="cpu")
except (pickle.UnpicklingError, RuntimeError):
    warnings.warn("Reloading with weights_only=False", RuntimeWarning)
    state = torch.load(ckpt, map_location="cpu", weights_only=False)

model_params, _ = state
IDX_XYZ, IDX_STATIC = 1, 19

def dump(tensor, path):
    np.savetxt(path, tensor.detach().cpu().view(-1, 3).numpy(),
               fmt="%.7g", delimiter=" ")
    print(f"{path.name} saved")

dump(model_params[IDX_XYZ], xyz_out)

if len(model_params) > IDX_STATIC:
    dump(model_params[IDX_STATIC], stat_out)
else:
    print("static_xyz not found in this checkpoint")


/var/folders/83/b6jh2lr54796xh55_hnd7gmr0000gn/T/ipykernel_65820/3980412749.py:10: RuntimeWarning: Reloading with weights_only=False
  warnings.warn("Reloading with weights_only=False", RuntimeWarning)


chkpnt6000.xyz.txt saved
chkpnt6000.static_xyz.txt saved


## Structuer of model_params (4D Gaussian)

### Main 4D Gaussian Attributes
- [0]  `active_sh_degree`
- [1]  `_xyz`
- [2]  `_features_dc`
- [3]  `_features_rest`
- [4]  `_scaling`
- [5]  `_rotation`
- [6]  `_opacity`
- [7]  `max_radii2D`
- [8]  `xyz_gradient_accum`
- [9]  `t_gradient_accum`
- [10] `denom`
- [11] `opt_dict`
- [12] `spatial_lr_scale`

### Time & Rotation Parameters
- [13] `_t`
- [14] `_scaling_t`
- [15] `_rotation_r`
- [16] `rot_4d`
- [17] `env_map` *(None, not used)*

### Temporal SH & Mode
- [18] `active_sh_degree_t`

### Static Frame Gaussian Info (e.g., t = 0)
- [19] `static_xyz`
- [20] `static_features_dc`
- [21] `static_features_rest`
- [22] `static_scaling`
- [23] `static_rotation`
- [24] `static_opacity`
- [25] `static_max_radii2D`
- [26] `static_denom`
- [27] `static_xyz_gradient_accum`
